# **Titanic - Machine Learning from Disaster**

**Atividade Capitulo 6 (Introducao ao Machine Learning)**

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
#importada as bibliotecas pedidas (1.(a-d))

%matplotlib inline
#exibe os graficos sem abrir outras janela (1.(e))
np.random.seed(0)
#fixa a semente aleatoria (1.(f))

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

# Download latest version
path = kagglehub.competition_download('titanic')
#comando que apareceu logo quando baixa os dados
#so copiar e colar

print("Path to competition files:", path)
print('Setup Complete')

# **6.1 Importacao e visualizacao do banco de dados**

In [ ]:
test_filepath='/kaggle/input/competitions/titanic/test.csv'
test_original=pd.read_csv(test_filepath)
train_filepath='/kaggle/input/competitions/titanic/train.csv'
train_original=pd.read_csv(train_filepath)
#importado os dados e lidos os dataframes (2.(a))
train_original.describe()
#gerar estatisticas (2.(b))

In [ ]:
train_original.head(20)

In [ ]:
train_original.info() #visualizar tipos de dados contidos (3.(a))
train_original.isnull().sum #(3.(b))

In [ ]:
train=train_original.select_dtypes('number')
test=test_original.select_dtypes('number')
#criar novos df de treino e test com somente colunas numericas
#(3.(c))
test=test.set_index('PassengerId').fillna(0)
#substitui os valores faltantes no df de teste para 
#zeros (3.(e))
train=train.set_index('PassengerId').dropna()
#retira as linhas que contenham informacoes
#faltantes (3.(d))
train

# **6.2 Exploracao do Banco de Dados**

In [ ]:
lim=[0, 10, 20, 30, 40, 50, 60, 70, 80]
labels=['(0-10)', '(10-20)', '(20-30)', '(30-40)', '(40-50)', '(50-60)', '(60-70)', '(70-80)']
#criada lista com rotulos (1.(a))
train['AgeGroup']=pd.cut(train['Age'], bins=lim, labels=labels)
#cria uma nova coluna que separa os passageiros em grupos por idade
#(1(b-c))
train.head(20)

In [ ]:
sns.histplot(x=train['AgeGroup'], hue=train['Survived'], multiple='fill')
#usado o histplot para permitir o emplilhamento em escala das duas barras, facilitando a verificacao
plt.title('Analise dos Dados Referentes a Idade')
plt.ylabel('proporcao entre sobreviventes e falecidos')
plt.xlabel('Grupos Etarios')
plt.legend(title='Resultado', labels=['Sobreviveu','Falececeu'])
#criado um grafico histplot que permite facil visualizacao sobre
#o resultado da trajedia (3.(d-e))
plt.show()

**Como, para a visualizacao manual dos dados, o quanto a pessoa pagou na passagem e qual o tipo de passagem que ela comprou representam a mesma coisa, nao sera feito o grafico da coluna fare**

In [ ]:
plt.figure(figsize=(4,4))
#definido o tamanho do grafico que sera criado
sns.histplot(x=train['Pclass'], hue=train['Survived'], multiple='fill')
plt.title('Analise dos Dados Referentes a Classe da Viagem')
plt.ylabel('proporcao entre sobreviventes e falecidos')
plt.xlabel('Grupos por Classe da Passagem')
plt.legend(title='Resultado', labels=['Sobreviveu','Falececeu'])
plt.xticks([1,2,3])
plt.show()

In [ ]:
fig, axes=plt.subplots(1,2, figsize=(6,6), sharey=True)
#permite a criacao de dois graficos lado a lado
fig.suptitle('Analise dos Dados Referentes a Quantidade de Familiares')
#titulo dos dois graficos, pois abordam parametros similares

#caracteristicas do grafico 1
sns.histplot(x=train['SibSp'], hue=train['Survived'], multiple='fill', ax=axes[0])
axes[0].set_ylabel('proporcao entre sobreviventes e falecidos')
axes[0].set_xlabel('Numero de irmaos e esposas a bordo')
axes[0].legend(title='Resultado', labels=['Sobreviveu','Falececeu'])

#caracteristicas do grafico 2
sns.histplot(x=train['Parch'], hue=train['Survived'], multiple='fill', ax=axes[1], legend=False)
#retirada a legenda pois ambos os graficos funcionam com apenas uma legenda
axes[1].set_ylabel('proporcao entre sobreviventes e falecidos')
axes[1].set_xlabel('Numero de pais e filhos a bordo')

plt.show()


**Grafico De Grupo Etarios:**
**As criancas foram as que mais sobreviveram, fruto do esforco dos pais e da equipe do barco para preservar as criancas, os adultos permaneceram no geral com uma porcentangem de sobrevivencia similar entre si, e os idosos sobreviveram consideravelmente menos, devido a desvantagem fisica.**



**Grafico de Classe da viagem:**
**Quanto maior a classe em que o passageiro viajou, maior era a chance dele sobreviver, provavelmente resultado do posionamento mais favoravel das cabines das classes mais altas, mas tambem o elitismo e preconceito presente na tripulacao da epoca**



**Graficos de parentes:**
**Mostra que, no geral, quanto maior o numero de parentes que voce tinha a bordo, menor era a chance de voce sobreviver, provavelmente devido ao desespero de tentar salvar a familia toda. Entretanto esses dados devem ser levados em consideracao com cautela, devido ao pouco numero de pessoas com muitos parentes a bordo, tornando a analise enviesada deivido ao baixo espaco amostral, alem disso, na epoca, era bem mais comum as familias com condicao financeira pior serem familias maiores no geral, entao esses dados podem acabar se misturando na hora de fazer a analise puramente sobre os dados sobre o tamanho da familia**


# **6.3 Treinamento do modelo e submissao das predicoes**

In [ ]:
X=train_original.select_dtypes('number')
X=X.drop(columns=['PassengerId'])
#selecionado apenas as caracteristicas numericas e
#retirada a coluna de identificacao do passageiro (1(a))
X

In [ ]:
y=X['Survived']
#criada a variavel que contem apenas 
#a coluna Survived (1(b))
X=X.drop(columns=['Survived'])
X

In [ ]:
from sklearn.model_selection import train_test_split
train_X, val_X, train_y, val_y =train_test_split(X, y,
                                                 test_size=0.2,
                                                 random_state=0) 
#usa o train test split com o parametro que delimita qual sera o tamanho
#de test e consequentemente do val (1(c))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
#importa o modelo e o verificador de acerto (2(a-b))

In [ ]:
model = RandomForestClassifier(random_state=1,)
model.fit(train_X, train_y)
#treinado o modelo com os dados de treino (2(c))
preds = model.predict(val_X)
#realizado e predicao dos valores
accuracy = accuracy_score(val_y, preds)
print(accuracy)
#(2(d-e))

**O modelo desenvolvido chegou a uma precisao de aproximadamente 72%, um valor bom gerado a partir dos dados que foram usados para treinar o modelo, entretanto, ainda tem outros dados (os que nao sao numeros) que ainda podem ser analisados, para gerar uma previsao mais precisa, com destaque especial a classificacao por genero, que provavelmente tera muito impacto devido ao habito de querer salvar mulheres e criancas primeiro**

# **Geracao do Modelo Definitivo (6.3.3 Geracao a submissao da predicao)**

In [ ]:
train_original.head(50)


In [ ]:
#ja tem o valor correto atribuido a variavel "y"
infos=['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
#separados as colunas que contem informacoes importantes para o
#treinamento do modelo 
X=train_original[infos].copy()
#comando . copy() adicionado para remover um aviso
#que podia estar prejjujjdicando o dataframe original
#atribui esses valores ao dataframe
X['Sex'] = X['Sex'].map({'male': 0, 'female': 1})
#transforma o texto da coluna de genero para numeros, 
#para que o moodelo possa avaliar
X['Embarked'] = X['Embarked'].map({'S': 0, 'C': 1, 'Q':2})
#tranforma o texto da coluna da cidade de embarque para 
#numeros, para que o modelo possa avaliar
X['Embarked'] = X['Embarked'].fillna(X['Embarked'].mode()[0])
#substitui os valores ausentes na coluna de cidade para a moda 
#da cidade de embarque, para nao precisar remover colunas devido
#a esses dados
X['Age'] = X['Age'].fillna(X['Age'].median())
X['Fare'] = X['Fare'].fillna(X['Fare'].mean())

X['FamSize'] = X['SibSp'] +X['Parch']
#X.head(50)

In [ ]:
train_X, val_X, train_y, val_y =train_test_split(X, y,
                                                 test_size=0.2,
                                                 random_state=0) 
model = RandomForestClassifier(random_state=1,
                              max_depth=5)
model.fit(train_X, train_y)
preds = model.predict(val_X)
accuracy = accuracy_score(val_y, preds)
print(accuracy)

In [ ]:
test_filepath='/kaggle/input/competitions/titanic/test.csv'
test=pd.read_csv(test_filepath)
test.head()

In [ ]:
testf = test[infos].copy()

testf['Sex'] = testf['Sex'].map({'male': 0, 'female': 1})

testf['Embarked'] = testf['Embarked'].map({'S': 0, 'C': 1, 'Q':2})

testf['Embarked'] = testf['Embarked'].fillna(testf['Embarked'].mode()[0])

testf['Fare'] = testf['Fare'].fillna(testf['Fare'].mean())
#substitui os valores ausentes na coluna de cidade para a moda 
#da cidade de embarque, para nao precisar remover colunas devido
#a esses dados
testf['Age'] = testf['Age'].fillna(testf['Age'].median())
testf['FamSize'] = testf['SibSp'] +testf['Parch']
#testf


In [ ]:
preds = model.predict(testf)
ids = test['PassengerId']
sub = pd.DataFrame({
    'PassengerId': ids,
    'Survived': preds
})
sub.head()

In [ ]:
sub.to_csv('submission.csv', index = False)
print('Arquivo criado com sucesso')